# Dataset preparation findings

Evidence behind the changes to `preprocess/preperator.py` and `train-whisper.py`.

The work started from one blocker - `ivrit-ai/crowd-transcribe-v5` could not be loaded
for training at all - and each fix below is here because measuring the data forced it.
Every number in this notebook is produced by the cell above it.

| # | Finding | Change it justifies |
|---|---------|---------------------|
| 1 | `crowd-transcribe-v5` stores its transcript in `sentence`, not `transcript` | `normalize_transcript_column` |
| 2 | `sentence` is the human-corrected text; `orig_sentence` is the machine output | alias ordering |
| 3 | 82 of 84 blank transcripts are mislabeled **speech**, not silence | `_select_transcribable_entries` |
| 4 | Most knesset timestamps **cannot be stripped** | `forced_ratio` in the ratio conversion |
| 5 | The ratio formula used the timestamp target for *both* attributes | per-attribute targets |
| 6 | Cross-over segments emitted `<\|notimestamps\|>` *with* timestamp tokens | `labels_carry_timestamps` |

## Setup

In [1]:
import os, sys, re, glob, json
from pathlib import Path

# Work from the repo root so `preprocess.preperator` imports resolve
root = Path.cwd()
while not (root / "train-whisper.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
os.chdir(root)

import numpy as np
import pyarrow.ipc as ipc
import datasets
from datasets import load_dataset
from transformers import WhisperProcessor

datasets.disable_progress_bars()
datasets.utils.logging.set_verbosity_error()

from preprocess.preperator import DatasetPreparator

MODEL = "ivrit-ai/whisper-large-v3"
processor = WhisperProcessor.from_pretrained(MODEL, language="hebrew", task="transcribe")
NOTS = processor.tokenizer.convert_tokens_to_ids("<|notimestamps|>")
TS_BEGIN = NOTS + 1

print("repo root:", root)
print("model    :", MODEL)
print("HF_HOME  :", os.environ.get("HF_HOME"))

/Users/ofir/Projects/asr-training/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repo root: /Users/ofir/Projects/asr-training
model    : ivrit-ai/whisper-large-v3
HF_HOME  : /Users/Shared/hf-home


## 1. The blocker: `crowd-transcribe-v5` cannot be prepared

The preparator requires the transcript in a column named `transcript`. This dataset
calls it `sentence`, so preparation fails before any audio is touched.

In [2]:
ct = load_dataset("ivrit-ai/crowd-transcribe-v5", split="train")
kn = load_dataset("ivrit-ai/knesset-plenums-whisper-training", split="train")

print("crowd-transcribe-v5 columns:", list(ct.features))
print("knesset columns            :", list(kn.features))
print()

prep = DatasetPreparator(processor)
try:
    prep.prepare_dataset(ct.select(range(4)))
    print("unexpectedly succeeded")
except ValueError as e:
    print(f"ValueError: {e}")

crowd-transcribe-v5 columns: ['uuid', 'audio', 'orig_sentence', 'sentence', 'is_retranscribe', 'transcriber', 'extra_data']
knesset columns            : ['audio', 'transcript', 'metadata', 'has_prev', 'has_timestamps', 'prev_transcript']

ValueError: Dataset must contain a 'transcript' feature


### The fix

`normalize_transcript_column` in `train-whisper.py` resolves a small set of aliases.
`rename_column` is metadata-only, so this copies no data. It also makes `eval-d1`,
`saspeech` (`text`) and `fleurs` (`transcription`) usable as `--eval_datasets`, which
they were not before.

In [3]:
import importlib.util
spec = importlib.util.spec_from_file_location("tw", root / "train-whisper.py")
tw = importlib.util.module_from_spec(spec); spec.loader.exec_module(tw)

for s in ["ivrit-ai/crowd-transcribe-v5:train[:5]",
          "ivrit-ai/knesset-plenums-whisper-training:train[:5]",
          "ivrit-ai/eval-d1:test[:5]",
          "upai-inc/saspeech:test[:5]"]:
    d = tw.load_datasets([s])[0]
    print(f"  {s:52s} -> transcript present: {'transcript' in d.features}")

ivrit-ai/crowd-transcribe-v5: using column 'sentence' as 'transcript'
ivrit-ai/crowd-transcribe-v5: quality filter dropped 0 of 5 rows
  ivrit-ai/crowd-transcribe-v5:train[:5]               -> transcript present: True
  ivrit-ai/knesset-plenums-whisper-training:train[:5]  -> transcript present: True
ivrit-ai/eval-d1: using column 'text' as 'transcript'
  ivrit-ai/eval-d1:test[:5]                            -> transcript present: True
upai-inc/saspeech: using column 'text' as 'transcript'
  upai-inc/saspeech:test[:5]                           -> transcript present: True


## 2. Which text column is the label?

`crowd-transcribe-v5` is crowd-*correction*: a model transcribed the audio, volunteers
fixed it. `orig_sentence` is the machine output, `sentence` the human-corrected text.
Picking the wrong one would train on the very errors the dataset exists to correct.

In [4]:
tbl = ct.select_columns(["sentence", "orig_sentence", "extra_data"]).data.table
sent = tbl.column("sentence").to_pylist()
orig = tbl.column("orig_sentence").to_pylist()
ed   = tbl.column("extra_data").to_pylist()

same = sum(1 for a, b in zip(sent, orig) if a == b)
print(f"rows                        : {len(sent):,}")
print(f"sentence == orig_sentence   : {same:,} ({same/len(sent):.1%})")
print(f"human actually changed it   : {len(sent)-same:,} ({1-same/len(sent):.1%})")
print(f"sentence == extra_data.text : {sum(1 for s,e in zip(sent,ed) if s==e['text']):,} (duplicate column)")
print()
shown = 0
for s, o in zip(sent, orig):
    if s != o and s.strip() and shown < 3:
        print(f"  machine: {o[:88]}")
        print(f"  human  : {s[:88]}")
        print()
        shown += 1

rows                        : 203,827
sentence == orig_sentence   : 95,457 (46.8%)
human actually changed it   : 108,370 (53.2%)
sentence == extra_data.text : 203,827 (duplicate column)

  machine: לא אני קטונתי כן אבל אני במקומו הייתי הולך על מהלך אמרתי לך ללכת כזה לאקדמיה של
  human  : לא, זה לא אני קטונתי כן אבל אני במקומו הייתי הולך על מהלך אמרתי לך ללכת כזה לאקדמיה של

  machine: ‫יותר סביר שהדברים האלה ‫נאמרו למה דברים אחרים.
  human  : ‫יותר סביר שהדברים האלה ‫נאמרו לעומת דברים אחרים.

  machine: ומחכים שגנב יגיע, ואז אתם תגיד לו, מה אתה עושה פה, אני אתפוס אותך, אני אעשה לך נו נו נו,
  human  : ומחכים שגנב יגיע, ואז אתם תגידו לו, מה אתה עושה פה, אני אתפוס אותך, אני אעשה לך נו נו נו



## 3. Are the blank transcripts useful "no speech" examples?

A fair objection to filtering blank transcripts: Whisper hallucinates on non-speech, and
training on empty targets is a known way to suppress that. So the question is not whether
no-speech examples are valuable - they are - but **what these particular rows contain**.

If the machine transcript is also empty, the segment is plausibly silent. If the machine
produced fluent text and the human left the box blank, the audio contains speech and the
label is simply missing.

In [6]:
from IPython.display import Audio, display

blank_idx = [i for i, s in enumerate(sent) if not s or not s.strip()]
orig_blank = sum(1 for i in blank_idx if not orig[i] or not orig[i].strip())

print(f"blank transcripts: {len(blank_idx)} of {len(sent):,}  ({len(blank_idx)/len(sent):.3%})")
print()
print(f"  machine ALSO heard nothing (plausibly silence): {orig_blank}")
print(f"  machine produced text, human left it blank    : {len(blank_idx)-orig_blank}")
print()
print("  what the machine heard on rows the human left blank:")
shown = 0
for i in blank_idx:
    if orig[i] and orig[i].strip() and shown < 6:
        display(Audio(ct[i]["audio"]["array"], rate=ct[i]["audio"]["sampling_rate"]))
        print(f"    [{ed[i]['duration']:5.2f}s] {orig[i][:76]}")
        shown += 1

blank transcripts: 84 of 203,827  (0.041%)

  machine ALSO heard nothing (plausibly silence): 2
  machine produced text, human left it blank    : 82

  what the machine heard on rows the human left blank:


    [ 2.38s] ‫אז כל זה עושה...


    [ 2.80s] תודה רבה


    [ 5.59s] השלב השני, אתה אומר איך אני יכול לכתוב את המערכת הזאת בצורת סט של משוואות?


    [ 2.95s] תודה רבה


    [ 6.24s] בסופו של דבר אתה לא יכול לחטוף ככל יכולתך, אתה יכול לקחת סמם מרץ ולהישאר ער 


    [ 2.90s] תודה רבה


**82 of 84 contain clear speech** - "תודה רבה" (thank you very much), a complete
technical sentence. Training on these teaches the model to emit nothing when it hears
common phrases, which is the deletion/truncation failure mode: the opposite of the
intended effect. Only 2 are plausibly silence.

Two further checks confirm these are not a no-speech signal worth keeping.

In [7]:
# (a) Does either dataset actually contain genuine no-speech segments?
ts_re = re.compile(r"<\|\d+\.\d+\|>")
shards = sorted(glob.glob(str(Path(os.environ["HF_HOME"]) /
    "datasets/ivrit-ai___knesset-plenums-whisper-training/default/0.0.0/*/*train-*.arrow")))[:4]

kn_blank = kn_tsonly = kn_total = 0
kn_transcripts = []
for f in shards:
    with open(f, "rb") as fh:
        t = ipc.open_stream(fh).read_all()
    for s in t.column("transcript").to_pylist():
        kn_total += 1
        kn_transcripts.append(s)
        if not s or not s.strip():
            kn_blank += 1
        elif not ts_re.sub("", s).strip():
            kn_tsonly += 1

print(f"knesset, {kn_total:,} rows sampled:")
print(f"  truly blank                : {kn_blank}")
print(f"  timestamp-only (silence)   : {kn_tsonly}")
print()
print(f"(b) blank share of crowd-transcribe: {len(blank_idx)/len(sent):.3%}")
print("    -> too small to shape hallucination behaviour even if all were real silence")

knesset, 4,536 rows sampled:
  truly blank                : 0
  timestamp-only (silence)   : 0

(b) blank share of crowd-transcribe: 0.041%
    -> too small to shape hallucination behaviour even if all were real silence


Knesset has **no** blank and **no** timestamp-only segments, so there is no no-speech
signal in these datasets to preserve. `_select_transcribable_entries` therefore drops
only blank strings, and deliberately **keeps** timestamp-only transcripts - a silent
segment carrying timestamps is legitimate training signal, if one ever appears.

Real anti-hallucination training would need no-speech examples to be *constructed*
deliberately at a controlled rate, which is a separate feature from this filter.

In [8]:
import inspect
print(inspect.getsource(DatasetPreparator._select_transcribable_entries))

    def _select_transcribable_entries(self, dataset):
        """Drop examples whose transcript is blank.

        Those tokenize down to a bare prefix + end-of-transcript sequence, which teaches
        the model to emit nothing for audible speech. Dropping them before the feature
        extraction map also avoids preparing audio we would never train on - scoping the
        filter to the transcript column keeps the audio undecoded, so this is nearly free.
        A transcript holding only timestamp tokens is kept: a silent segment is
        legitimate training signal.
        """
        return dataset.filter(
            lambda transcript: bool(transcript and transcript.strip()), input_columns="transcript"
        )



## 4. Most knesset timestamps cannot be stripped

`_is_removable_timestamp_token_ids` refuses to strip timestamps from *cross-over*
segments - where the final segment continues past the audio slice, so the text is only
correct with timestamps present. Those examples always carry timestamps no matter what
is sampled.

This is the fact the sampling maths was missing.

In [ ]:
p = DatasetPreparator(processor)
removable = []
for s in kn_transcripts:
    ids = processor.tokenizer(s, add_special_tokens=False, add_prefix_space=False,
                              return_attention_mask=False)["input_ids"]
    removable.append(p._is_removable_timestamp_token_ids(ids))
removable = np.array(removable)

forced = 1 - removable.mean()
print(f"knesset, {len(removable):,} rows:")
print(f"  has_timestamps           : 1.000")
print(f"  removable (strippable)   : {removable.mean():.3f}")
print(f"  FORCED - cross-over      : {forced:.3f}")
print()
print(f"  => the timestamped share can never go below {forced:.1%}")

knesset, 4,536 rows:
  has_timestamps           : 1.000
  removable (strippable)   : 0.274
  FORCED - cross-over      : 0.726

  => the timestamped share can never go below 72.6%


## 5. The ratio conversion was wrong in two independent ways

The original code:

```python
relative_sampling_ratios = {
    attr: (
        min(1.0, self.timestamp_sample_prob / attr_estimation["estimated_ratio"])
        if attr_estimation["estimated_ratio"] > 0
        else 1.0
    )
    for attr, attr_estimation in estimations.items()
}
```

**(a)** `timestamp_sample_prob` is the numerator for *both* attributes, so
`--include_prev_text_prob` never reached the data - it was dead.

**(b)** `target / frequency` assumes examples without the attribute can never gain it.
For timestamps that is false: the cross-over examples are attribute-*forced*, not
attribute-free. And when the frequency is 0 the fallback of `1.0` forces the attribute
onto **every** example - which is what made synthetic injection fire 100% of the time.

In [10]:
def old_ratio(timestamp_prob, freq):
    """The pre-fix formula, reproduced here for comparison."""
    return min(1.0, timestamp_prob / freq) if freq > 0 else 1.0

R = removable.mean()      # measured in section 4, not hardcoded
F = forced
print(f"knesset timestamps (removable={R:.3f}, forced={F:.3f})")
print(f"{'target':>7} {'old ratio':>10} {'old share':>10} {'new ratio':>10} {'new share':>10}")
for t in (0.1, 0.5, 0.8, 0.9, 1.0):
    o = old_ratio(t, R)
    n = DatasetPreparator._relative_sampling_ratio(t, R, forced_ratio=F)
    print(f"{t:>7} {o:>10.3f} {F+R*o:>10.3f} {n:>10.3f} {F+R*n:>10.3f}")

print()
print("crowd-transcribe timestamps (removable=0.0, forced=0.0) - injection rate")
for t in (0.3, 0.5):
    o = old_ratio(t, 0.0)
    n = DatasetPreparator._relative_sampling_ratio(t, 0.0, forced_ratio=0.0)
    print(f"  target={t}: old={o:.3f} (injects on every example)   new={n:.3f}")

print()
print("prev-text on knesset (has_prev=0.913, never forced)")
for t in (0.2, 0.5):
    o = old_ratio(0.5, 0.913)   # old always used timestamp_sample_prob = 0.5
    n = DatasetPreparator._relative_sampling_ratio(t, 0.913)
    print(f"  target={t}: old={o:.3f} (ignored the flag)   new={n:.3f}")

knesset timestamps (removable=0.274, forced=0.726)
 target  old ratio  old share  new ratio  new share
    0.1      0.365      0.826      0.000      0.726
    0.5      1.000      1.000      0.000      0.726
    0.8      1.000      1.000      0.271      0.800
    0.9      1.000      1.000      0.635      0.900
    1.0      1.000      1.000      1.000      1.000

crowd-transcribe timestamps (removable=0.0, forced=0.0) - injection rate
  target=0.3: old=1.000 (injects on every example)   new=0.300
  target=0.5: old=1.000 (injects on every example)   new=0.500

prev-text on knesset (has_prev=0.913, never forced)
  target=0.2: old=0.548 (ignored the flag)   new=0.219
  target=0.5: old=0.548 (ignored the flag)   new=0.548


The new conversion. `forced_ratio` already covers part of the target, so only the
sampleable remainder has to make it up:

In [11]:
print(inspect.getsource(DatasetPreparator._relative_sampling_ratio))

    @staticmethod
    def _relative_sampling_ratio(
        target_prob: float, sampled_ratio: float, forced_ratio: float = 0.0
    ) -> float:
        """Convert a target rate over the whole dataset into a per-example sampling rate.

        An attribute is only sampled on the examples where the choice actually exists.
        Examples carrying it unconditionally (`forced_ratio`) already cover part of the
        target, so the sampled ones only have to make up the remainder -
        `(target_prob - forced_ratio) / sampled_ratio`, clamped to a probability.

        A target below `forced_ratio` is unreachable: the ratio clamps to zero and the
        attribute lands on the forced share. When nothing is sampleable at all, the ratio
        is meaningless for the strip/keep decision, so we fall back to the target itself -
        that keeps augmentations which synthesize the attribute (see
        `inject_synthetic_timestamps`) firing at the requested rate rather than on every
        

## 6. Cross-over segments produced self-contradictory labels

The prefix was chosen from the *sampling intent*:

```python
with_timestamps = has_timestamps and should_train_on_timestamps
```

But stripping can be refused. When `should_train_on_timestamps` is False and the segment
is a cross-over, the tokens keep their timestamps while the prefix says
`<|notimestamps|>` - the label contradicts itself.

This was latent because the old ratio saturated to 1.0 on knesset (so the branch rarely
ran). Fixing finding 5 drives that ratio to 0 at the default target, which would have
fired this on every cross-over segment. Below, both rules are applied to real
examples - no audio needed, the disagreement is purely in the label.

In [16]:
crossover = [s for s, rem in zip(kn_transcripts, removable) if not rem][:400]
should_train_on_timestamps = False        # the case the bug needs

bad_old = bad_new = 0
example = None
for s in crossover:
    ids = processor.tokenizer(s, add_special_tokens=False, add_prefix_space=False,
                              return_attention_mask=False)["input_ids"]
    has_timestamps = True
    labels_carry_timestamps = has_timestamps
    if has_timestamps and not should_train_on_timestamps:
        if p._is_removable_timestamp_token_ids(ids):      # False for cross-over
            ids = [i for i in ids if i < TS_BEGIN]
            labels_carry_timestamps = False

    old_prefix_ts = has_timestamps and should_train_on_timestamps   # pre-fix rule
    new_prefix_ts = labels_carry_timestamps                         # post-fix rule
    tokens_have_ts = any(i >= TS_BEGIN for i in ids)

    if tokens_have_ts and not old_prefix_ts:
        bad_old += 1
        if example is None:
            example = processor.tokenizer.decode(
                p.prefix_tokens_no_ts + ids, skip_special_tokens=False)
    if tokens_have_ts and not new_prefix_ts:
        bad_new += 1

print(f"cross-over examples checked: {len(crossover)}"
      f"  ({forced:.1%} of knesset is cross-over)")
print(f"  contradictory under the OLD rule: {bad_old}/{len(crossover)}")
print(f"  contradictory under the NEW rule: {bad_new}/{len(crossover)}")
print()
print("  This sample is cross-over by construction, so the old rule fails on all of")
print("  them whenever should_train_on_timestamps is False. What changed is how often")
print("  that happens: fixing section 5 drops the ratio to 0 at the default target,")
print("  making it the common case rather than a rare one.")
print()
print("  what the old rule emitted:")
print(f"    {example}...")

cross-over examples checked: 400  (72.6% of knesset is cross-over)
  contradictory under the OLD rule: 400/400
  contradictory under the NEW rule: 0/400

  This sample is cross-over by construction, so the old rule fails on all of
  them whenever should_train_on_timestamps is False. What changed is how often
  that happens: fixing section 5 drops the ratio to 0 at the default target,
  making it the common case rather than a rare one.

  what the old rule emitted:
    <|startoftranscript|><|he|><|transcribe|><|notimestamps|> לרמוס את האמונה שבני-האדם, כל בני-האדם, נולדו בצלם. אבל, גברתי היושבת-ראש, ליום הזה יש תפקיד אחד נוסף: אין שום ספק שהיום שבו הקהילה הבין-לאומית החליטה להכיר...


## 7. End-to-end verification

Everything above is static analysis. These cells run the real preparation pipeline and
measure the resulting labels.

In [13]:
def measure(ds, **kw):
    out = DatasetPreparator(processor, **kw).prepare_dataset(ds)
    labels = out["labels"]
    timestamped = np.mean([any(t >= TS_BEGIN for t in r) for r in labels])
    contradictory = sum(1 for r in labels if NOTS in r and any(t >= TS_BEGIN for t in r))
    prev = np.mean([processor.tokenizer.convert_tokens_to_ids("<|startofprev|>") in r for r in labels])
    return timestamped, prev, contradictory, out.num_rows

kn_small = kn.select(range(120))
print("knesset (120 examples)")
for target in (0.5, 0.9):
    ts, prev, bad, n = measure(kn_small, timestamp_sample_prob=target, condition_on_prev_sample_prob=0.2)
    print(f"  timestamp target={target}: timestamped={ts:.3f}  prev={prev:.3f}  contradictory={bad}/{n}")

knesset (120 examples)
Estimating attribute frequencies for relative sub-sampling.
Requested timestamp share 0.50 is below the 0.67 of examples whose timestamps cannot be stripped - preparing at that floor instead.
Estimated attribute frequencies: {'has_removable_timestamps': 0.3333333333333333, 'has_forced_timestamps': 0.6666666666666666, 'has_prev': 0.9166666666666666}
Relative sampling ratios: {'has_removable_timestamps': 0.0, 'has_prev': 0.2181818181818182}
  timestamp target=0.5: timestamped=0.667  prev=0.242  contradictory=0/120
Estimating attribute frequencies for relative sub-sampling.
Estimated attribute frequencies: {'has_removable_timestamps': 0.3333333333333333, 'has_forced_timestamps': 0.6666666666666666, 'has_prev': 0.9166666666666666}
Relative sampling ratios: {'has_removable_timestamps': 0.7000000000000002, 'has_prev': 0.2181818181818182}
  timestamp target=0.9: timestamped=0.875  prev=0.183  contradictory=0/120


In [14]:
ct_small = tw.load_datasets(["ivrit-ai/crowd-transcribe-v5:train[:300]"])[0]
print("crowd-transcribe (300 examples), synthetic timestamp injection on")
for target in (0.3, 0.6):
    ts, prev, bad, n = measure(ct_small, timestamp_sample_prob=target,
                               condition_on_prev_sample_prob=0.5,
                               inject_synthetic_timestamps=True)
    print(f"  timestamp target={target}: injected={ts:.3f}  contradictory={bad}/{n}   (was 1.000 before)")

ivrit-ai/crowd-transcribe-v5: using column 'sentence' as 'transcript'
ivrit-ai/crowd-transcribe-v5: quality filter dropped 1 of 300 rows
crowd-transcribe (300 examples), synthetic timestamp injection on
Dataset does not contain a 'has_timestamps' feature. Assuming no timestamps.
Dataset does not contain a 'has_prev' feature. Assuming no previous transcript.
Estimating attribute frequencies for relative sub-sampling.
Estimated attribute frequencies: {'has_removable_timestamps': 0.0, 'has_forced_timestamps': 0.0, 'has_prev': 0.0}
Relative sampling ratios: {'has_removable_timestamps': 0.3, 'has_prev': 0.5}
  timestamp target=0.3: injected=0.341  contradictory=0/299   (was 1.000 before)
Dataset does not contain a 'has_timestamps' feature. Assuming no timestamps.
Dataset does not contain a 'has_prev' feature. Assuming no previous transcript.
Estimating attribute frequencies for relative sub-sampling.
Estimated attribute frequencies: {'has_removable_timestamps': 0.0, 'has_forced_timestamps':

`prev` tracks the requested 0.2 instead of following the timestamp probability, the
timestamped share honours the target where it is reachable and clamps to the cross-over
floor where it is not, injection lands on the requested rate rather than every example,
and no label contradicts itself.

## 8. Filtering cost on the full dataset

Both filters are scoped to the columns they read, which keeps the audio column undecoded.

In [15]:
import time
t0 = time.time()
full = tw.load_datasets(["ivrit-ai/crowd-transcribe-v5:train"])[0]
final = DatasetPreparator(processor)._select_transcribable_entries(full)
elapsed = time.time() - t0

print(f"  203,827 -> {full.num_rows:,} (dataset quality flags) -> {final.num_rows:,} (blank transcripts)")
print(f"  dropped {203827-final.num_rows} rows total ({(203827-final.num_rows)/203827:.3%}) in {elapsed:.1f}s")
print("  no audio decoded")

ivrit-ai/crowd-transcribe-v5: using column 'sentence' as 'transcript'
ivrit-ai/crowd-transcribe-v5: quality filter dropped 70 of 203827 rows
  203,827 -> 203,757 (dataset quality flags) -> 203,673 (blank transcripts)
  dropped 154 rows total (0.076%) in 7.3s
  no audio decoded


## Summary

| Change | File | Justified by |
|--------|------|--------------|
| `normalize_transcript_column` | `train-whisper.py` | §1 - the dataset was unusable |
| alias order prefers `sentence` | `train-whisper.py` | §2 - `orig_sentence` is the machine output |
| `DatasetRowFilter` registry | `train-whisper.py` | §8 - per-dataset rules, audio left undecoded |
| `_select_transcribable_entries` | `preperator.py` | §3 - 82/84 blanks are mislabeled speech |
| `forced_ratio` in ratio conversion | `preperator.py` | §4, §5 - most timestamps are forced |
| per-attribute targets | `preperator.py` | §5 - `--include_prev_text_prob` was dead |
| `labels_carry_timestamps` | `preperator.py` | §6 - contradictory labels |

Measured effect on the defaults:

| Setting | Before | After |
|---------|--------|-------|
| knesset, `--include_timestamps_prob 0.5` | 100% timestamped, silently | the cross-over floor + warning |
| knesset, `--include_timestamps_prob 0.9` | 100% | 90% |
| knesset, `--include_prev_text_prob 0.2` | ~90%, flag ignored | 20% |
| crowd-transcribe + injection, target 0.3 | 100% injected | 30% |
| self-contradictory labels | present | none |